# 🏆 PLAN 1: PERFORMANCE — Tesla T4 (16GB VRAM • ~1.53 CU/h)

> **Purpose:** High-performance configuration for daily software engineering and architecture design (~1.53 CU/h, recommended when CU > 35).
> *(Adaptive behavior: Runs optimized Page Cache loading on Tesla T4 16GB; automatically unlocks Dual-Resident `OLLAMA_MAX_LOADED_MODELS="2"` when switched to NVIDIA A100 40GB).*

---

### 🧩 Model Allocation (Conductor & Worker Protocol)

| Role | Model | VRAM | Primary Responsibilities |
| :--- | :--- | :--- | :--- |
| 🎼 **Conductor** | **`deepseek-r1:14b`** | ~9.0 GB | Requirements analysis, reasoning `<think>`, emits checklist `.agents/kb/HANDOFF.md` |
| 🛠️ **Worker** | **`qwen2.5-coder:14b` ⭐** | ~9.5 GB | Executes task pipeline (Code, Unit Tests, Workflow CLI/JSON, Mermaid Diagrams) |

---

### 🛡️ Memory & VRAM Architecture (T4 16GB)
- **Configuration:** `OLLAMA_MAX_LOADED_MODELS="1"`, `OLLAMA_FLASH_ATTENTION="1"`, `OLLAMA_KV_CACHE_TYPE="q8_0"`.
- **Peak VRAM (16K context):** `13.0 GB / 15.84 GB` (82% VRAM ceiling).
- **Buffer & Virtual Memory:** 16GB NVMe Swapfile enabled with warmed-up Linux OS Page Cache; 1 single model transition after planning step (~1.8s) with KV Cache Prefix reuse across coding tasks.

---

💡 **Related Notebooks:**
- 🚀 **A100 40GB Dual-Resident (32B):** [`colab_ollama_plan1_performance_a100.ipynb`](colab_ollama_plan1_performance_a100.ipynb)
- ⚡ **T4 Dual-Resident Fast (CU < 35 or 65–75 tok/s):** [`colab_ollama_plan2_fast_t4.ipynb`](colab_ollama_plan2_fast_t4.ipynb)
- 📘 **Basic Tutorial (1-Model):** [`colab_ollama_tutorial.ipynb`](colab_ollama_tutorial.ipynb)

### Step 1: Hardware & VRAM Verification (NVIDIA GPU)
Verify the allocated NVIDIA GPU and active VRAM capacity (Tesla T4 16GB or A100 40GB) on Google Colab.

In [ ]:
# Step 1: Verify NVIDIA GPU hardware and active VRAM capacity
!nvidia-smi

### Step 2: Model Storage Configuration (Colab NVMe SSD or Google Drive)
Configure weight storage destination:
- **`USE_GDRIVE = False` (Recommended):** Store on internal Colab NVMe SSD (`/root/.ollama/models`) for maximum throughput (>600 MB/s) and zero Google Drive authorization prompts.
- **`USE_GDRIVE = True`:** Mount and persist cache to Google Drive for multi-session reuse across browser tabs.

In [ ]:
import os

# Set to True if running in a web browser (Safari/Chrome) and you wish to persist models to Google Drive
# Default is False to prioritize high-speed internal NVMe SSD (>600 MB/s) & 100% compatibility with IDE Kernels
USE_GDRIVE = False

if USE_GDRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        models_dir = '/content/drive/MyDrive/ollama_models'
        os.environ['OLLAMA_MODELS'] = models_dir
        os.makedirs(models_dir, exist_ok=True)
        print(f"✅ Google Drive cache activated at: {models_dir}")
    except Exception as e:
        print(f"ℹ️ Running in IDE Kernel (Skipping Google Drive mount: {e}).")
        print("⚡ Using internal Colab NVMe SSD storage (/root/.ollama/models).")
else:
    print("⚡ Using internal Colab NVMe SSD storage (/root/.ollama/models) — Optimal model loading speed.")

### Step 3: Install Ollama & Launch Daemon
Install Ollama, configure Linux NVMe Swapfile, and start background daemon:
- **16GB NVMe Swapfile:** Expands total virtual memory on Colab T4 to 28.67 GB, preventing OOM during model switching.
- **VRAM Optimizations:** Enables `OLLAMA_FLASH_ATTENTION="1"` and `OLLAMA_KV_CACHE_TYPE="q8_0"`.
- **Hardware Auto-Detection:** Dynamically sets `OLLAMA_MAX_LOADED_MODELS="1"` on T4 16GB (or `"2"` on A100 40GB).

In [ ]:
import os
import subprocess
import time
import urllib.request
from IPython.display import clear_output

# 1. Provision 16GB Linux NVMe Swapfile (Expands virtual memory to 28.67 GB, 100% OOM protection)
!test -f /swapfile || (fallocate -l 16G /swapfile && chmod 600 /swapfile && mkswap /swapfile && swapon /swapfile)

# 2. Detect GPU VRAM to dynamically configure OLLAMA_MAX_LOADED_MODELS
vram_mb = 16000
gpu_name = "Tesla T4"
try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
        text=True
    ).strip().splitlines()[0]
    gpu_name, mem_str = [x.strip() for x in out.split(",")]
    vram_mb = int(float(mem_str))
except Exception:
    pass

max_loaded = "2" if vram_mb >= 30000 else "1"

# 3. Install zstd and official Ollama Linux release
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

# 4. Configure Plan 1: Performance environment variables
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_KEEP_ALIVE"] = "24h"                 # Keep model resident in VRAM / RAM Cache throughout the session
env["OLLAMA_MAX_LOADED_MODELS"] = max_loaded     # "1" on T4 16GB (82% VRAM ceiling), "2" if running on A100 40GB
env["OLLAMA_FLASH_ATTENTION"] = "1"              # Enable Flash Attention for faster prefill and decode
env["OLLAMA_KV_CACHE_TYPE"] = "q8_0"             # Quantize KV Cache to 8-bit, saving 50% VRAM for 16K context

# 5. Launch 'ollama serve' in background
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
clear_output(wait=True)
time.sleep(3)

try:
    with urllib.request.urlopen("http://localhost:11434/") as resp:
        print(f"🎉 Plan 1 Performance Daemon ready on {gpu_name} ({vram_mb/1024:.1f} GB VRAM)!")
        print(f"⚙️ Config: OLLAMA_MAX_LOADED_MODELS={max_loaded} | FLASH_ATTENTION=1 | KV_CACHE=q8_0 | 16GB Swap")
        print(resp.read().decode("utf-8"))
except Exception as e:
    print("⚠️ Failed to connect to Ollama daemon:", e)

### Step 4: Pull Plan 1 Dual Models & Warm-up Linux Page Cache
Download models and warm up memory cache:
1. **`deepseek-r1:14b` (Conductor):** Requirements analysis and compact handoff generation (`.agents/kb/HANDOFF.md`).
2. **`qwen2.5-coder:14b` ⭐ (Worker):** Specialized code generation, test implementation, and architecture diagramming.
3. **Cache Warm-up:** Pre-load model weights into Linux Page Cache/Swap and pre-pin `qwen2.5-coder:14b` into VRAM.

In [ ]:
import json
import urllib.request
from IPython.display import clear_output

PLAN1_T4_MODELS = [
    ("deepseek-r1:14b",   "🎼 Model 1 (Conductor): DeepSeek CoT <think> Planning & Reasoning (~9.0 GB)"),
    ("qwen2.5-coder:14b", "🛠️ Model 2 (Worker Specialist ⭐): Coding, Workflow & Mermaid Diagram (~9.5 GB)"),
]

print("⏳ Pulling Plan 1: Performance model pair (Tesla T4)...")
for tag, role in PLAN1_T4_MODELS:
    print(f"\n📦 Pulling {tag} — {role}...")
    !ollama pull {tag}

print("\n🔥 Warming up Linux OS Page Cache & pre-pinning Model 2 (qwen2.5-coder:14b)...")
for tag, _ in PLAN1_T4_MODELS:
    try:
        payload = json.dumps({"model": tag, "prompt": "hi", "stream": False, "keep_alive": "24h"}).encode("utf-8")
        req = urllib.request.Request("http://localhost:11434/api/generate", data=payload, headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=120):
            print(f"  ✨ Successfully warmed up cache: {tag}")
    except Exception as e:
        print(f"  ⚠️ Skipping warm-up for {tag}: {e}")

clear_output(wait=True)
print("✅ Successfully pulled and warmed up Plan 1 models (Performance T4)!")
!ollama list

### Step 5: Expose Cloudflare Quick Tunnel & Connect to IDE
Establish an HTTPS tunnel forwarding `localhost:11434` to a public `*.trycloudflare.com` URL:
- **Tunnel URL:** Auto-extracted URL for configuring into your MCP client or IDE.
- **Pipe Drainer:** Background thread continuously flushes `cloudflared` stdout to prevent 64KB OS pipe buffer exhaustion and tunnel lockups (HTTP 524).

In [ ]:
import re
import subprocess
import threading
import time
from IPython.display import clear_output

# 1. Download and install Cloudflare Tunnel (cloudflared)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 2. Launch cloudflared forwarding HTTPS to localhost:11434
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# 3. Read stdout to extract public '*.trycloudflare.com' URL
tunnel_url = None
print("⏳ Establishing Cloudflare Quick Tunnel...")
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.2)

# 4. CRITICAL: Run background thread to drain stdout continuously and prevent 64KB OS pipe buffer lockups
def _drain_tunnel_logs(proc):
    try:
        for _ in iter(proc.stdout.readline, ""):
            pass
    except Exception:
        pass

threading.Thread(target=_drain_tunnel_logs, args=(tunnel_proc,), daemon=True).start()

clear_output(wait=True)

if tunnel_url:
    print("=" * 78)
    print("🎉 PLAN 1: PERFORMANCE (T4 16GB) TUNNEL IS ONLINE!")
    print(f"🔗 PUBLIC ENDPOINT URL: {tunnel_url}")
    print("=" * 78)
    print("\n🚀 METHOD 1 (Paste endpoint URL directly into IDE / CLI chat):\n")
    print(f"   {tunnel_url}")
    print("   (The agent will automatically invoke `ollama_set_endpoint`, verify connectivity, and route calls to Colab GPU!)\n")
    print("⚡ METHOD 2 (Run one-line command in project terminal):\n")
    print(f"   python3 colab_ollama_mcp/server.py --set-endpoint {tunnel_url}")
    print("\n" + "-" * 78)
else:
    print("❌ Failed to obtain Cloudflare Tunnel URL. Please re-run this cell.")

### Step 6: Session Keep-Alive (HTML5 Silent Audio Loop)
Activates an HTML5 silent audio element to maintain *Media Active* state in the browser tab, preventing Google Colab from disconnecting or idling in the background.

In [ ]:
from IPython.display import HTML, display

display(HTML('''
<div style="padding: 14px; background: #e8f5e9; border: 1px solid #4caf50; border-radius: 8px; font-family: sans-serif;">
  <b style="color: #2e7d32;">🟢 Keep-Alive Active (Silent Audio Loop)</b>
  <p style="margin: 4px 0 8px 0; color: #1b5e20; font-size: 13px;">
     Keeps your Google Colab session active and responsive while coding in your IDE.
  </p>
  <audio loop autoplay controls style="height: 30px;">
    <source src="data:audio/wav;base64,UklGRsQAAABXQVZFZm10IBAAAAABAAEAQB8AAIA+AAACABAAZGF0YaAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA" type="audio/wav">
  </audio>
</div>
'''))

### Step 7: Real-Time Resource Telemetry Dashboard (`/api/ps`)
Monitor GPU VRAM, System RAM, Swap, Disk, and loaded models in VRAM:
- **`AUTO_REFRESH = False` (Default):** Instant single snapshot of current system resources.
- **`AUTO_REFRESH = True`:** Auto-refreshing dashboard every 5 seconds (`clear_output(wait=True)`).

In [ ]:
import json
import shutil
import subprocess
import time
import urllib.request
from IPython.display import clear_output

AUTO_REFRESH = False
REFRESH_SECONDS = 5

def make_bar(pct: float, width: int = 24) -> str:
    filled = max(0, min(width, int(round((pct / 100.0) * width))))
    return "█" * filled + "░" * (width - filled)

def print_telemetry() -> None:
    print("=" * 78)
    print(f"📊 PLAN 1 PERFORMANCE (T4) TELEMETRY — {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}")
    print("=" * 78)

    # 1. GPU VRAM, Load & Temperature
    try:
        out = subprocess.check_output([
            "nvidia-smi",
            "--query-gpu=name,temperature.gpu,utilization.gpu,memory.used,memory.total",
            "--format=csv,noheader,nounits",
        ], text=True).strip()
        for line in out.splitlines():
            name, temp, util, mem_used, mem_total = [x.strip() for x in line.split(",")]
            used_gb = float(mem_used) / 1024.0
            total_gb = float(mem_total) / 1024.0
            vram_pct = (float(mem_used) / float(mem_total)) * 100.0 if float(mem_total) > 0 else 0.0
            util_pct = float(util)
            print(f"🎮 GPU Hardware : {name}  (🌡️ {temp}°C)")
            print(f"⚡ GPU Core Load: [{make_bar(util_pct)}] {util_pct:5.1f}%")
            print(f"🧠 GPU VRAM     : [{make_bar(vram_pct)}] {vram_pct:5.1f}%  ({used_gb:.2f} GB / {total_gb:.2f} GB)")
    except Exception as e:
        print(f"⚠️ GPU Status   : {e}")

    # 2. System RAM, Page Cache & Swapfile
    try:
        meminfo = {}
        with open("/proc/meminfo", "r") as f:
            for line in f:
                parts = line.split(":")
                if len(parts) == 2:
                    meminfo[parts[0].strip()] = int(parts[1].strip().split()[0])
        total_ram_gb = meminfo.get("MemTotal", 0) / (1024 ** 2)
        avail_ram_gb = meminfo.get("MemAvailable", 0) / (1024 ** 2)
        used_ram_gb = total_ram_gb - avail_ram_gb
        ram_pct = (used_ram_gb / total_ram_gb * 100.0) if total_ram_gb > 0 else 0.0
        cached_gb = meminfo.get("Cached", 0) / (1024 ** 2)
        swap_total_gb = meminfo.get("SwapTotal", 0) / (1024 ** 2)
        swap_free_gb = meminfo.get("SwapFree", 0) / (1024 ** 2)
        swap_used_gb = swap_total_gb - swap_free_gb
        print(f"💻 System RAM   : [{make_bar(ram_pct)}] {ram_pct:5.1f}%  ({used_ram_gb:.2f} GB / {total_ram_gb:.2f} GB | Page Cache: {cached_gb:.2f} GB)")
        print(f"🛡️ NVMe Swapfile: {swap_used_gb:.2f} GB / {swap_total_gb:.2f} GB (Total Virtual Memory: {total_ram_gb + swap_total_gb:.2f} GB)")
    except Exception as e:
        print(f"⚠️ System RAM   : {e}")

    # 3. Colab SSD Disk Usage
    try:
        total_d, used_d, _ = shutil.disk_usage("/")
        used_d_gb = used_d / (1024 ** 3)
        total_d_gb = total_d / (1024 ** 3)
        disk_pct = (used_d / total_d * 100.0) if total_d > 0 else 0.0
        print(f"💾 Colab Disk   : [{make_bar(disk_pct)}] {disk_pct:5.1f}%  ({used_d_gb:.1f} GB / {total_d_gb:.1f} GB)")
    except Exception as e:
        print(f"⚠️ Colab Disk   : {e}")

    # 4. Active Models Pinned in GPU VRAM (/api/ps)
    print("-" * 78)
    try:
        with urllib.request.urlopen("http://localhost:11434/api/ps", timeout=5) as resp:
            ps_data = json.loads(resp.read().decode("utf-8"))
            running = ps_data.get("models", [])
            if running:
                print(f"🔥 Currently pinned in GPU VRAM ({len(running)} model{'s' if len(running) > 1 else ''}):")
                for m in running:
                    m_name = m.get("name", "unknown")
                    vram_gb = m.get("size_vram", 0) / (1024 ** 3)
                    total_gb = m.get("size", 0) / (1024 ** 3)
                    gpu_offload = (vram_gb / total_gb * 100.0) if total_gb > 0 else 100.0
                    print(f"   • {m_name} — VRAM: {vram_gb:.2f} GB ({gpu_offload:.0f}% GPU offload)")
            else:
                print("💤 GPU VRAM is currently idle (Models load on-demand on first request)")
    except Exception as e:
        print(f"🔴 Ollama Status: Offline or unresponsive ({e})")
    print("=" * 78)

try:
    while True:
        clear_output(wait=True)
        print_telemetry()
        if not AUTO_REFRESH:
            break
        print(f"🔄 Auto-refreshing every {REFRESH_SECONDS}s... (Click Stop ⏹️ to halt)")
        time.sleep(REFRESH_SECONDS)
except KeyboardInterrupt:
    print("⏹️ Telemetry dashboard stopped.")